In [4]:
import sympy as sp
import numpy as np
import math
import sys

def run_bisection():
    print("=== CHƯƠNG TRÌNH GIẢI PHƯƠNG TRÌNH BẰNG PHƯƠNG PHÁP CHIA ĐÔI ===")
    
    try:
        # --- 0. Nhập liệu ---
        raw_f_str = input("Nhập hàm f(x) (ví dụ: x**3 - x - 1): ")
        f_str = raw_f_str.replace('^', '**') 
        
        a = float(input("Nhập đầu mút a: "))
        b = float(input("Nhập đầu mút b: "))
        
        # Tự động sửa lỗi nếu nhập a > b
        if a > b:
            a, b = b, a
            
        epsilon = float(input("Nhập sai số mục tiêu epsilon (ví dụ: 1e-4): "))
        precision = int(input("Nhập số chữ số thập phân hiển thị (ví dụ: 5): "))

        # --- 1. Xử lý toán học với SymPy ---
        x = sp.Symbol('x')
        f_expr = sp.sympify(f_str)
        f = sp.lambdify(x, f_expr, 'numpy')

        # --- 2. Kiểm tra điều kiện và Đánh giá tiên nghiệm ---
        fa, fb = f(a), f(b)
        
        # Bắt lỗi nghiệm nằm ngay tại mút
        if np.isclose(fa, 0):
            print(f"\n=> KẾT QUẢ NGAY LẬP TỨC: Nghiệm chính là tại mút a = {a}")
            return
        if np.isclose(fb, 0):
            print(f"\n=> KẾT QUẢ NGAY LẬP TỨC: Nghiệm chính là tại mút b = {b}")
            return

        print("\n" + "="*60)
        print("### 1. Kiểm tra điều kiện và Đánh giá tiên nghiệm")
        print(f"* **Hàm số:** $f(x) = {sp.latex(f_expr)}$")
        print(f"* **Khoảng phân li:** $f({a}) = {fa:.{precision}f}$, $f({b}) = {fb:.{precision}f}$")
        
        if fa * fb > 0:
            print(f"\n[!] LỖI: $f({a}) \\cdot f({b}) > 0$. Hàm không đổi dấu ở hai đầu mút. Không đảm bảo có nghiệm trên đoạn này.")
            return
            
        print(f"=> $f({a}) \\cdot f({b}) < 0$. Đảm bảo tồn tại ít nhất 1 nghiệm trên $[{a}, {b}]$.")

        # Tính toán N (Tiên nghiệm)
        L = b - a
        N = math.ceil(math.log2(L / epsilon))
        
        print(f"\n* **Công thức số bước lặp tiên nghiệm:**")
        print(f"  $$n \\ge \\log_2 \\left( \\frac{{b-a}}{{\\epsilon}} \\right)$$")
        # Đã sửa lỗi "n" bằng cách dùng "{{n}}" ở đây:
        print(f"  * *Sự liên kết:* Chiều dài khoảng cách ly ban đầu là $b-a$. Sai số ở bước $n$ (khoảng cách từ điểm giữa đến nghiệm) là $\\frac{{b-a}}{{2^{{n}}}}$. Đặt ngưỡng này $\\le \\epsilon$, ta tính được số bước.")
        print(f"=> Cần lặp tối đa **$N = {N}$** bước để đảm bảo sai số $\\Delta_n \\le {epsilon:.1e}$.")

        # --- 3. Quá trình lặp ---
        history = []
        curr_a, curr_b = a, b

        for n in range(1, N + 1):
            c = (curr_a + curr_b) / 2
            fc = f(c)
            
            # Sai số: nửa chiều dài khoảng hiện tại
            err = (curr_b - curr_a) / 2
            
            step_data = {
                'n': n, 'a': curr_a, 'b': curr_b, 'c': c, 
                'f(a)': f(curr_a), 'f(c)': fc, 'err': err, 'action': ''
            }
            
            if np.isclose(fc, 0, atol=1e-15):
                step_data['action'] = "Nghiệm đúng"
                history.append(step_data)
                break
                
            if f(curr_a) * fc < 0:
                step_data['action'] = "b = c"
                history.append(step_data)
                curr_b = c
            else:
                step_data['action'] = "a = c"
                history.append(step_data)
                curr_a = c

            # Dừng sớm nếu đánh giá hậu nghiệm đạt (đôi khi nhanh hơn N dự đoán)
            if err <= epsilon:
                break

        total_steps = len(history)

        # --- 4. Hiển thị chi tiết các bước ---
        print("\n### 2. Chi tiết các bước lặp tiêu biểu")
        print(f"**Công thức:** $c_n = \\frac{{a_{{n-1}} + b_{{n-1}}}}{{2}}$, $\\Delta_n = \\frac{{b_{{n-1}} - a_{{n-1}}}}{{2}}$\n")
        
        # 2 BƯỚC ĐẦU
        print("**Hai bước lặp đầu tiên:**")
        for i in range(min(2, total_steps)):
            step = history[i]
            print(f"* **Bước $n = {step['n']}$:** Khoảng $[{step['a']:.{precision}f}, {step['b']:.{precision}f}]$")
            print(f"  $$c_{{{step['n']}}} = \\frac{{{step['a']:.{precision}f} + {step['b']:.{precision}f}}}{{2}} = {step['c']:.{precision}f}$$")
            print(f"  $$\\Delta_{{{step['n']}}} = {step['err']:.{precision}g}$$")
            print(f"  * $f(c) = {step['f(c)']:.{precision}f} {('< 0' if step['f(c)'] < 0 else '> 0')}$ $\\Rightarrow$ Chọn {step['action']}")

        # 2 BƯỚC CUỐI
        if total_steps > 2:
            print("\n**Hai bước lặp cuối cùng:**")
            for i in range(max(2, total_steps - 2), total_steps):
                step = history[i]
                print(f"* **Bước $n = {step['n']}$:** Khoảng $[{step['a']:.{precision}f}, {step['b']:.{precision}f}]$")
                print(f"  $$c_{{{step['n']}}} = \\frac{{{step['a']:.{precision}f} + {step['b']:.{precision}f}}}{{2}} = {step['c']:.{precision}f}$$")
                print(f"  $$\\Delta_{{{step['n']}}} = {step['err']:.{precision}g}$$")
                print(f"  * $f(c) = {step['f(c)']:.{precision}f} {('< 0' if step['f(c)'] < 0 else '> 0')}$ $\\Rightarrow$ Chọn {step['action']}")

        # --- 5. Bảng lặp rút gọn ---
        print("\n### 3. Bảng lặp tổng hợp rút gọn\n")
        print(f"| $n$ | $a_{{n-1}}$ | $b_{{n-1}}$ | $c_n$ | $f(c_n)$ | $\\Delta_n$ | Cập nhật |")
        print("| :--- | :--- | :--- | :--- | :--- | :--- | :--- |")
        
        for i in range(min(3, total_steps)):
            s = history[i]
            print(f"| {s['n']} | {s['a']:.{precision}f} | {s['b']:.{precision}f} | {s['c']:.{precision}f} | {s['f(c)']:.{precision}f} | {s['err']:.{precision}g} | {s['action']} |")
            
        if total_steps > 6:
            print("| ... | ... | ... | ... | ... | ... | ... |")
            
        if total_steps > 3:
            for i in range(max(3, total_steps - 3), total_steps):
                s = history[i]
                print(f"| {s['n']} | {s['a']:.{precision}f} | {s['b']:.{precision}f} | {s['c']:.{precision}f} | {s['f(c)']:.{precision}f} | {s['err']:.{precision}g} | {s['action']} |")

        final_c = history[-1]['c']
        final_err = history[-1]['err']
        print(f"\n=> **KẾT LUẬN:** Qua {total_steps} bước lặp, nghiệm xấp xỉ là **$x^* \\approx {final_c:.{precision}f}$** (Sai số $\\Delta = {final_err:.{precision}g} \\le {epsilon:.1e}$).")

    except Exception as e:
        print(f"\n[!] CÓ LỖI XẢY RA: {e}")

if __name__ == "__main__":
    run_bisection()
    input("\nNhấn Enter để thoát chương trình...")

=== CHƯƠNG TRÌNH GIẢI PHƯƠNG TRÌNH BẰNG PHƯƠNG PHÁP CHIA ĐÔI ===

### 1. Kiểm tra điều kiện và Đánh giá tiên nghiệm
* **Hàm số:** $f(x) = x^{3} + 3 x^{2} - 1$
* **Khoảng phân li:** $f(0.1) = -0.9690000$, $f(1.0) = 3.0000000$
=> $f(0.1) \cdot f(1.0) < 0$. Đảm bảo tồn tại ít nhất 1 nghiệm trên $[0.1, 1.0]$.

* **Công thức số bước lặp tiên nghiệm:**
  $$n \ge \log_2 \left( \frac{b-a}{\epsilon} \right)$$
  * *Sự liên kết:* Chiều dài khoảng cách ly ban đầu là $b-a$. Sai số ở bước $n$ (khoảng cách từ điểm giữa đến nghiệm) là $\frac{b-a}{2^{n}}$. Đặt ngưỡng này $\le \epsilon$, ta tính được số bước.
=> Cần lặp tối đa **$N = 24$** bước để đảm bảo sai số $\Delta_n \le 1.0e-07$.

### 2. Chi tiết các bước lặp tiêu biểu
**Công thức:** $c_n = \frac{a_{n-1} + b_{n-1}}{2}$, $\Delta_n = \frac{b_{n-1} - a_{n-1}}{2}$

**Hai bước lặp đầu tiên:**
* **Bước $n = 1$:** Khoảng $[0.1000000, 1.0000000]$
  $$c_{1} = \frac{0.1000000 + 1.0000000}{2} = 0.5500000$$
  $$\Delta_{1} = 0.45$$
  * $f(c) = 0.0738750 > 0$ $